# Beyond Backscatter: GRD/GEE Coherence Estimation

This public notebook estimates InSAR coherence from two detected Sentinel-1 GRD SAR backscatter images. It uses Sentinel-1 GRD data from Google Earth Engine and the GRD/GEE Beyond Backscatter model workflow.

This notebook does not use the SLC inference path. The inputs are detected GRD backscatter images, not complex SLC data.

The RGB products below are SAR/coherence pseudo-RGB visualizations derived from SAR amplitudes and predicted coherence. They are not true optical images and are not optical reconstructions.

## 1. Public Colab Setup

When this notebook is opened directly from GitHub in Colab, the repository is not automatically installed as a Python package. This cell clones the public repository, installs the small Colab requirements file, and exposes the helper modules on `sys.path`.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

GITHUB_REPO_URL = "https://github.com/FPSica/BeyondBackscatter.git"
GITHUB_BRANCH = "main"
REPO_DIR = "/content/BeyondBackscatter"

repo_dir = Path(REPO_DIR)
if IN_COLAB:
    if not (repo_dir / ".git").exists():
        if repo_dir.exists() and any(repo_dir.iterdir()):
            raise RuntimeError(f"{repo_dir} exists but is not a git checkout. Choose a different REPO_DIR.")
        subprocess.check_call(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(repo_dir)])
    else:
        subprocess.check_call(["git", "fetch", "origin"], cwd=repo_dir)
        subprocess.check_call(["git", "checkout", GITHUB_BRANCH], cwd=repo_dir)
        subprocess.check_call(["git", "pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=repo_dir)
else:
    cwd = Path.cwd().resolve()
    if (cwd / "src" / "colab_grd_gee").exists():
        repo_dir = cwd
    elif not (repo_dir / "src" / "colab_grd_gee").exists():
        raise FileNotFoundError("Run this notebook from the repository root or use Colab so the repo can be cloned.")

requirements_path = repo_dir / "requirements-colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])

for path in [repo_dir, repo_dir / "src"]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

for module_name in [
    "colab_grd_gee.gee_s1",
    "colab_grd_gee.io_utils",
    "colab_grd_gee.model_loader",
    "colab_grd_gee.tf_model",
    "colab_grd_gee.inference",
    "colab_grd_gee.visualization",
]:
    importlib.import_module(module_name)

print(f"Running in Colab: {IN_COLAB}")
print(f"Repository ready: {repo_dir}")

## 2. User Configuration

Edit this single cell first. Keep credentials and tokens out of the notebook. For private Hugging Face models, use Colab secrets or environment variables such as `HF_TOKEN`.

In [ ]:
from pathlib import Path

GEE_PROJECT_ID = "your-gee-project-id"

POLARIZATION = "VV"
ORBIT_PASS = "ASCENDING"
RELATIVE_ORBIT = ""   # Optional; set to "", None, or a number-like string.
INSTRUMENT_MODE = "IW"

DATE1_START = "2026-05-01"
DATE1_END   = "2026-05-30"
DATE2_START = "2026-06-01"
DATE2_END   = "2026-06-30"

SCALE_METERS = 10
CRS = "EPSG:4326"

MODEL_SOURCE = "huggingface"  # options: "huggingface", "google_drive", "local"
MODEL_FRAMEWORK = "tensorflow"

HF_REPO_ID = "FPSica/beyond-backscatter-grd-gee"
HF_REVISION = "main"
HF_WEIGHTS_FILENAME = "model.weights.h5"
HF_CONFIG_FILENAME = "config.yaml"

LOCAL_MODEL_DIR = "/content/BeyondBackscatter/model"
GDRIVE_MODEL_DIR = "/content/drive/MyDrive/back2coh_grd_gee_model"

OUTPUT_DIR = Path(repo_dir) / "outputs"
GEE_INPUT_DIR = OUTPUT_DIR / "gee_inputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEE_INPUT_DIR.mkdir(parents=True, exist_ok=True)

relative_orbit = None if RELATIVE_ORBIT in (None, "") else int(RELATIVE_ORBIT)
print(f"Outputs will be written under: {OUTPUT_DIR}")

## 3. Earth Engine Authentication

This cell uses the standard Earth Engine flow:

```python
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)
```

No credentials are stored in the notebook.

In [ ]:
import ee

if not GEE_PROJECT_ID or GEE_PROJECT_ID == "your-gee-project-id":
    raise ValueError("Set GEE_PROJECT_ID to your Earth Engine enabled Google Cloud project ID.")

try:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception as exc:
    raise RuntimeError(
        "Earth Engine authentication failed. Check that your account has Earth Engine access "
        "and that GEE_PROJECT_ID is a valid Cloud project enabled for Earth Engine."
    ) from exc

print("Earth Engine initialized.")

## 4. ROI Selection

Choose one ROI method:

- `ROI_METHOD = "draw"`: draw a rectangle or polygon on the interactive map.
- `ROI_METHOD = "polygon"`: use manual lon/lat polygon coordinates.
- `ROI_METHOD = "geojson"`: paste a GeoJSON geometry or feature.

Start with a small ROI for the first run to keep Earth Engine download and model inference fast.

In [ ]:
import geemap
from colab_grd_gee import gee_s1

ROI_METHOD = "draw"  # options: "draw", "polygon", "geojson"

MANUAL_POLYGON_LONLAT = [
    [-122.60, 36.88],
    [-121.03, 36.88],
    [-121.03, 37.96],
    [-122.60, 37.96],
]

MANUAL_GEOJSON = {
    "type": "Polygon",
    "coordinates": [[
        [-122.60, 36.88], [-121.03, 36.88], [-121.03, 37.96], [-122.60, 37.96]
    ]],
}

Map = geemap.Map(center=[48.08, 11.31], zoom=10)
Map.add_basemap("SATELLITE")
Map

In [ ]:
if ROI_METHOD == "draw":
    roi = gee_s1.get_drawn_roi(Map)
elif ROI_METHOD == "polygon":
    roi = gee_s1.polygon_from_lonlat(MANUAL_POLYGON_LONLAT)
elif ROI_METHOD == "geojson":
    roi = gee_s1.geometry_from_geojson(MANUAL_GEOJSON)
else:
    raise ValueError("ROI_METHOD must be 'draw', 'polygon', or 'geojson'.")

Map.addLayer(roi, {}, "Selected ROI")
Map.centerObject(roi, 12)
roi_geojson = roi.getInfo()
print("ROI selected.")
Map

## 5. Sentinel-1 GRD Search And Selection

The notebook queries `COPERNICUS/S1_GRD` using the configured ROI, polarization, orbit pass, optional relative orbit number, instrument mode, and date windows. Review the acquisition tables before downloading.

In [ ]:
from colab_grd_gee import gee_s1

records_t1, count_t1, collection_t1 = gee_s1.list_acquisitions(
    roi=roi,
    start_date=DATE1_START,
    end_date=DATE1_END,
    polarization=POLARIZATION,
    orbit_pass=ORBIT_PASS,
    relative_orbit=relative_orbit,
    instrument_mode=INSTRUMENT_MODE,
)
records_t2, count_t2, collection_t2 = gee_s1.list_acquisitions(
    roi=roi,
    start_date=DATE2_START,
    end_date=DATE2_END,
    polarization=POLARIZATION,
    orbit_pass=ORBIT_PASS,
    relative_orbit=relative_orbit,
    instrument_mode=INSTRUMENT_MODE,
)

print(f"Window 1: {DATE1_START} to {DATE1_END}: {count_t1} image(s)")
print(f"Window 2: {DATE2_START} to {DATE2_END}: {count_t2} image(s)")

try:
    import pandas as pd
    print("Acquisitions for window 1")
    display(pd.DataFrame.from_records(records_t1))
    print("Acquisitions for window 2")
    display(pd.DataFrame.from_records(records_t2))
except Exception:
    print(records_t1)
    print(records_t2)

if count_t1 == 0 or count_t2 == 0:
    raise RuntimeError("At least one date window has no matching Sentinel-1 GRD images. Adjust dates or filters.")

rel1 = {r["relative_orbit"] for r in records_t1 if r.get("relative_orbit") is not None}
rel2 = {r["relative_orbit"] for r in records_t2 if r.get("relative_orbit") is not None}
platform1 = {r["platform"] for r in records_t1 if r.get("platform") is not None}
platform2 = {r["platform"] for r in records_t2 if r.get("platform") is not None}
if RELATIVE_ORBIT in (None, "") and rel1 and rel2 and rel1.isdisjoint(rel2):
    print("Warning: the two windows do not share a relative orbit. Consider setting RELATIVE_ORBIT explicitly.")
if platform1 and platform2 and platform1.isdisjoint(platform2):
    print("Warning: the two windows use different Sentinel-1 platforms.")

## 6. GEE Preprocessing And Download

The Earth Engine preprocessing mirrors the reference JavaScript workflow:

- use `COPERNICUS/S1_GRD`;
- filter by ROI, `instrumentMode`, `orbitProperties_pass`, optional relative orbit, and date window;
- use `.median()` for each date window;
- convert dB sigma0 to linear sigma0 with `pow(10, db / 10)`;
- select the requested polarization;
- clip to ROI;
- download at the configured scale and CRS.

Files are saved under `outputs/gee_inputs/`.

In [ ]:
from colab_grd_gee import io_utils

sigma0_t1_tif, sigma0_t2_tif = gee_s1.download_sigma0_pair(
    roi=roi,
    output_dir=GEE_INPUT_DIR,
    date1_start=DATE1_START,
    date1_end=DATE1_END,
    date2_start=DATE2_START,
    date2_end=DATE2_END,
    polarization=POLARIZATION,
    orbit_pass=ORBIT_PASS,
    relative_orbit=relative_orbit,
    instrument_mode=INSTRUMENT_MODE,
    scale=SCALE_METERS,
    crs=CRS,
)

metadata = {
    "collection": "COPERNICUS/S1_GRD",
    "roi_geojson": roi_geojson,
    "filters": {
        "polarization": POLARIZATION,
        "orbit_pass": ORBIT_PASS,
        "relative_orbit": relative_orbit,
        "instrument_mode": INSTRUMENT_MODE,
        "date1_start": DATE1_START,
        "date1_end": DATE1_END,
        "date2_start": DATE2_START,
        "date2_end": DATE2_END,
        "scale_meters": SCALE_METERS,
        "crs": CRS,
    },
    "selected_image_ids": {
        "date_window_1": [r["id"] for r in records_t1],
        "date_window_2": [r["id"] for r in records_t2],
    },
    "preprocessing": {
        "composite": "median",
        "db_to_linear": "pow(10, db / 10)",
        "clip_to_roi": True,
    },
    "files": {
        "sigma0_t1_linear_tif": str(sigma0_t1_tif),
        "sigma0_t2_linear_tif": str(sigma0_t2_tif),
    },
}
metadata_path = io_utils.save_metadata_json(GEE_INPUT_DIR / "metadata.json", metadata)
print(f"Downloaded: {sigma0_t1_tif}")
print(f"Downloaded: {sigma0_t2_tif}")
print(f"Metadata:   {metadata_path}")

## 7. Model Loading

The model weights are not stored in GitHub. By default, this notebook downloads the real GRD/GEE TensorFlow/Keras model weights from Hugging Face Hub.

Expected model-repo files:

- `model.weights.h5`
- `config.yaml`
- `README.md`

The Keras ResUNet architecture is provided by this GitHub repository under `src/colab_grd_gee/tf_model.py`. The loader imports TensorFlow, enables GPU memory growth when available, builds the GRD/GEE model, first tries standard Keras weight loading, and falls back to legacy Keras H5 by-name loading for newer Keras runtimes.

In [ ]:
from colab_grd_gee import model_loader

# Optional for Google Drive models:
# from google.colab import drive
# drive.mount('/content/drive')

model_bundle = model_loader.load_model_bundle(
    source=MODEL_SOURCE,
    framework=MODEL_FRAMEWORK,
    hf_repo_id=HF_REPO_ID,
    hf_revision=HF_REVISION,
    weights_filename=HF_WEIGHTS_FILENAME,
    config_filename=HF_CONFIG_FILENAME,
    local_model_dir=LOCAL_MODEL_DIR,
    gdrive_model_dir=GDRIVE_MODEL_DIR,
)

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
except Exception:
    tf = None
    gpus = []

print(f"Model directory: {model_bundle.model_dir}")
print(f"Weights:         {model_bundle.weights_path}")
print(f"Framework:       {model_bundle.framework}")
print(f"Runtime:         {model_bundle.runtime}")
print(f"TensorFlow GPUs: {gpus}")

## 8. Input Preparation

This section reads the downloaded GeoTIFFs, validates alignment, builds a common valid mask, and applies the GRD/GEE model preprocessing. The preprocessing follows the original GRD/GEE path: linear sigma0 -> dB -> clip to `[-20, 0]` -> normalize to `[0, 1]` with channel order `[t1, t2]`, unless the Hugging Face `config.yaml` overrides those settings. The model is called with TensorFlow/Keras NHWC patches.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from colab_grd_gee import inference, io_utils, visualization

primary_linear, profile_t1, mask_t1, bounds_t1 = io_utils.read_single_band_geotiff(sigma0_t1_tif, fill_value=0.0)
secondary_linear, profile_t2, mask_t2, bounds_t2 = io_utils.read_single_band_geotiff(sigma0_t2_tif, fill_value=0.0)
io_utils.assert_aligned(profile_t1, profile_t2, primary_linear.shape, secondary_linear.shape)
valid_mask = mask_t1 | mask_t2 | ~np.isfinite(primary_linear) | ~np.isfinite(secondary_linear)

model_input, prep_debug = inference.linear_sigma0_to_model_inputs(primary_linear, secondary_linear, model_bundle.config)
assert model_input.dtype == np.float32
assert model_input.ndim == 3
assert model_input.shape[1:] == primary_linear.shape

np.save(OUTPUT_DIR / "sigma0_t1_linear.npy", primary_linear.astype(np.float32))
np.save(OUTPUT_DIR / "sigma0_t2_linear.npy", secondary_linear.astype(np.float32))

print(f"Input raster shape: {primary_linear.shape}")
print(f"Model tensor shape: {model_input.shape} [channels, rows, cols]")
print(f"Model tensor dtype: {model_input.dtype}")
print(f"Preprocessing config: {prep_debug['config']}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
axes[0].imshow(visualization.percentile_normalize(visualization.to_db(primary_linear), mask=valid_mask), cmap="gray")
axes[0].set_title("t1 sigma0 dB stretch")
axes[1].imshow(visualization.percentile_normalize(visualization.to_db(secondary_linear), mask=valid_mask), cmap="gray")
axes[1].set_title("t2 sigma0 dB stretch")
axes[2].imshow(model_input[0], cmap="gray", vmin=0, vmax=1)
axes[2].set_title("normalized model channel 0")
for ax in axes:
    ax.set_axis_off()
plt.show()

visualization.plot_histograms(primary_linear, secondary_linear, model_input)
plt.show()

## 9. Inference

This runs tiled TensorFlow/Keras GRD/GEE inference. Patch size, stride, batch size, and aggregation default to values from `config.yaml`, falling back to the original GRD/GEE defaults: 128 pixel patches, 32 pixel stride, and Kaiser-window aggregation.

In [ ]:
from colab_grd_gee import inference, io_utils

PATCH_SIZE = None  # None uses config/default
STRIDE = None      # None uses config/default
BATCH_SIZE = None  # None uses config/default

coherence, inference_debug = inference.predict_tiled(
    model_bundle=model_bundle,
    primary_linear=primary_linear,
    secondary_linear=secondary_linear,
    mask=valid_mask,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
)

coherence_npy = OUTPUT_DIR / "coherence_pred.npy"
coherence_tif = OUTPUT_DIR / "coherence_pred.tif"
coherence_png = OUTPUT_DIR / "coherence_pred.png"
np.save(coherence_npy, coherence.astype(np.float32))
io_utils.save_single_band_geotiff(coherence_tif, coherence, profile_t1, dtype="float32", nodata=-9999.0)

plt.figure(figsize=(7, 6))
im = plt.imshow(coherence, cmap="gray", vmin=0, vmax=1)
plt.colorbar(im, label="predicted coherence")
plt.axis("off")
plt.tight_layout()
plt.savefig(coherence_png, dpi=180, bbox_inches="tight")
plt.show()

print(f"Inference debug: {inference_debug}")
print(f"Saved: {coherence_npy}")
print(f"Saved: {coherence_tif}")
print(f"Saved: {coherence_png}")

## 10. Coherence Display

The predicted coherence map is clipped to `[0, 1]`, saved as NumPy, PNG, and GeoTIFF, and displayed with summary statistics.

In [ ]:
from colab_grd_gee import visualization

stats = visualization.coherence_stats(coherence)
print("Coherence statistics")
for key, value in stats.items():
    print(f"  {key}: {value}")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(coherence, cmap="gray", vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label="predicted coherence")
ax.set_title("Predicted InSAR coherence")
ax.set_axis_off()
plt.show()

try:
    result_map = geemap.Map()
    result_map.add_basemap("SATELLITE")
    result_map.add_raster(str(coherence_tif), cmap="gray", vmin=0, vmax=1, layer_name="Predicted coherence")
    result_map
except Exception as exc:
    print(f"Map overlay skipped: {exc}")

## 11. RGB Visualization

Two RGB products are created from SAR amplitudes/backscatter and predicted coherence.

Mode A, diagnostic false color:

- R = normalized amplitude/backscatter at t2
- G = predicted coherence
- B = normalized amplitude/backscatter at t1

Mode B, pseudo-natural SAR/coherence RGB:

- Uses average amplitude, amplitude difference, coherence, gamma correction, and robust percentile normalization.
- Aims to make water darker/blue-ish, decorrelated vegetation greener, stable bright/built-up areas gray/tan/white, and changed areas slightly warmer/redder.

These products are visualization aids only. They are not true optical imagery and not optical reconstructions.

In [ ]:
PERCENTILE_RANGE = (2.0, 98.0)
GAMMA = 0.95
WATER_STRENGTH = 0.22
GREEN_STRENGTH = 0.28
COHERENCE_BRIGHTNESS = 0.40
CHANGE_REDNESS = 0.20

diagnostic_rgb = visualization.make_diagnostic_rgb(
    primary_linear=primary_linear,
    secondary_linear=secondary_linear,
    coherence=coherence,
    mask=valid_mask,
    percentile_range=PERCENTILE_RANGE,
)
pseudo_natural_rgb = visualization.make_pseudo_natural_rgb(
    primary_linear=primary_linear,
    secondary_linear=secondary_linear,
    coherence=coherence,
    mask=valid_mask,
    percentile_range=PERCENTILE_RANGE,
    gamma=GAMMA,
    water_strength=WATER_STRENGTH,
    green_strength=GREEN_STRENGTH,
    coherence_brightness=COHERENCE_BRIGHTNESS,
    change_redness=CHANGE_REDNESS,
)

rgb_diag_png = OUTPUT_DIR / "rgb_diagnostic.png"
rgb_pseudo_png = OUTPUT_DIR / "rgb_pseudo_natural.png"
rgb_pseudo_tif = OUTPUT_DIR / "rgb_pseudo_natural.tif"
io_utils.save_png(rgb_diag_png, diagnostic_rgb)
io_utils.save_png(rgb_pseudo_png, pseudo_natural_rgb)
io_utils.save_rgb_geotiff(rgb_pseudo_tif, pseudo_natural_rgb, profile_t1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
axes[0].imshow(diagnostic_rgb)
axes[0].set_title("Diagnostic SAR/coherence false color")
axes[1].imshow(pseudo_natural_rgb)
axes[1].set_title("Pseudo-natural SAR/coherence RGB")
for ax in axes:
    ax.set_axis_off()
plt.show()

print(f"Saved: {rgb_diag_png}")
print(f"Saved: {rgb_pseudo_png}")
print(f"Saved: {rgb_pseudo_tif}")

## 12. Export Outputs

This cell zips the `outputs/` directory and optionally copies it to Google Drive.

In [ ]:
from colab_grd_gee import io_utils

zip_path = io_utils.zip_outputs(OUTPUT_DIR, OUTPUT_DIR / "beyond_backscatter_outputs.zip")
print(f"Created: {zip_path}")

COPY_TO_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/beyond_backscatter_outputs"

if COPY_TO_GOOGLE_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    drive_output_dir = Path(DRIVE_OUTPUT_DIR)
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_output_dir / zip_path.name)
    print(f"Copied zip to: {drive_output_dir / zip_path.name}")

## 13. Smoke Test Without Earth Engine Or Model Weights

This lightweight test exercises the RGB visualization functions only. It does not require Earth Engine credentials or model weights.

In [ ]:
import numpy as np
from colab_grd_gee import visualization

rng = np.random.default_rng(42)
fake_t1 = np.clip(rng.lognormal(mean=-2.0, sigma=0.5, size=(64, 64)), 1e-5, 1.0).astype(np.float32)
fake_t2 = np.clip(fake_t1 * rng.lognormal(mean=0.0, sigma=0.15, size=(64, 64)), 1e-5, 1.0).astype(np.float32)
fake_coh = np.clip(0.75 - 0.4 * np.abs(fake_t2 - fake_t1) + 0.05 * rng.normal(size=(64, 64)), 0, 1).astype(np.float32)

fake_diag = visualization.make_diagnostic_rgb(fake_t1, fake_t2, fake_coh)
fake_pseudo = visualization.make_pseudo_natural_rgb(fake_t1, fake_t2, fake_coh)

assert fake_diag.shape == (64, 64, 3)
assert fake_pseudo.shape == (64, 64, 3)
assert np.nanmin(fake_diag) >= 0 and np.nanmax(fake_diag) <= 1
assert np.nanmin(fake_pseudo) >= 0 and np.nanmax(fake_pseudo) <= 1
print("Smoke test passed: RGB visualization outputs are in [0, 1].")